In [2]:
import zipfile
import pandas as pd
import os

# Define the path to the zip file
zip_file_path = '/content/Student Performance.zip'

# Define the directory to extract to
extract_dir = '/content/student_performance_data'

# Create the extraction directory if it doesn't exist
os.makedirs(extract_dir, exist_ok=True)

# Unzip the file
with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
    zip_ref.extractall(extract_dir)
    print(f"Extracted contents to: {extract_dir}")
    # List contents of the extracted directory to find the CSV file
    extracted_files = os.listdir(extract_dir)
    print(f"Contents of extracted directory: {extracted_files}")

# Assuming there is a CSV file directly in the extracted directory
# We'll try to find the first CSV file or assume a common name
csv_file = None
for f in extracted_files:
    if f.endswith('.csv'):
        csv_file = os.path.join(extract_dir, f)
        break

if csv_file:
    # Load the dataset into a pandas DataFrame
    df = pd.read_csv(csv_file)

    # Display the first 5 rows of the DataFrame
    print("\nFirst 5 rows of the dataset:")
    print(df.head())

    # Display basic information about the dataset
    print("\nDataset Information:")
    df.info()
else:
    print("No CSV file found in the extracted directory.")

Extracted contents to: /content/student_performance_data
Contents of extracted directory: ['StudentsPerformance.csv']

First 5 rows of the dataset:
   gender race/ethnicity parental level of education         lunch  \
0  female        group B           bachelor's degree      standard   
1  female        group C                some college      standard   
2  female        group B             master's degree      standard   
3    male        group A          associate's degree  free/reduced   
4    male        group C                some college      standard   

  test preparation course  math score  reading score  writing score  
0                    none          72             72             74  
1               completed          69             90             88  
2                    none          90             95             93  
3                    none          47             57             44  
4                    none          76             78             75  

Dataset In

### Dataset Structure Overview

The previous output already showed `df.info()`. Now, let's look at the descriptive statistics for numerical columns and unique values for categorical columns.

In [3]:
# Display descriptive statistics for numerical columns
print("\nDescriptive Statistics for Numerical Columns:")
print(df.describe())


Descriptive Statistics for Numerical Columns:
       math score  reading score  writing score
count  1000.00000    1000.000000    1000.000000
mean     66.08900      69.169000      68.054000
std      15.16308      14.600192      15.195657
min       0.00000      17.000000      10.000000
25%      57.00000      59.000000      57.750000
50%      66.00000      70.000000      69.000000
75%      77.00000      79.000000      79.000000
max     100.00000     100.000000     100.000000


In [4]:
# Display unique values and their counts for categorical columns
print("\nValue Counts for Categorical Columns:")
for column in df.select_dtypes(include='object').columns:
    print(f"\n--- {column} ---")
    print(df[column].value_counts())
    print(f"Number of unique values: {df[column].nunique()}")


Value Counts for Categorical Columns:

--- gender ---
gender
female    518
male      482
Name: count, dtype: int64
Number of unique values: 2

--- race/ethnicity ---
race/ethnicity
group C    319
group D    262
group B    190
group E    140
group A     89
Name: count, dtype: int64
Number of unique values: 5

--- parental level of education ---
parental level of education
some college          226
associate's degree    222
high school           196
some high school      179
bachelor's degree     118
master's degree        59
Name: count, dtype: int64
Number of unique values: 6

--- lunch ---
lunch
standard        645
free/reduced    355
Name: count, dtype: int64
Number of unique values: 2

--- test preparation course ---
test preparation course
none         642
completed    358
Name: count, dtype: int64
Number of unique values: 2


### Data Cleaning and Preprocessing

In [6]:
import re
import pandas as pd
import os
import zipfile

# Check if 'df' DataFrame exists in the current scope.
# If not, it means the data loading cell wasn't run or kernel was restarted.
# To prevent NameError, we'll attempt to re-load the data.
if 'df' not in locals() and 'df' not in globals():
    print("Warning: 'df' DataFrame not found. Attempting to re-load data...")
    try:
        # Define the path to the zip file and extraction directory from the original data loading cell
        zip_file_path = '/content/Student Performance.zip'
        extract_dir = '/content/student_performance_data'

        # Create the extraction directory if it doesn't exist
        os.makedirs(extract_dir, exist_ok=True)

        # Check if the CSV file exists in the extracted directory, if not, re-extract
        csv_file_name = 'StudentsPerformance.csv'
        extracted_csv_path = os.path.join(extract_dir, csv_file_name)

        if not os.path.exists(extracted_csv_path):
            with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
                zip_ref.extractall(extract_dir)
            print(f"Data re-extracted to: {extract_dir}")

        # Load the dataset into a pandas DataFrame
        df = pd.read_csv(extracted_csv_path)
        print("DataFrame 'df' re-loaded successfully within this cell.")
    except Exception as e:
        print(f"Error re-loading 'df' within this cell: {e}")
        # As a fallback, create an empty DataFrame to prevent further NameErrors
        df = pd.DataFrame()
        print("Created an empty DataFrame as fallback.")

# Clean column names: replace spaces with underscores and convert to lowercase
def clean_column_names(df_to_clean):
    cols = df_to_clean.columns
    new_cols = []
    for col in cols:
        # Removed non-alphanumeric characters except underscore and spaces (which are replaced by underscore)
        new_col = re.sub(r'[^a-zA-Z0-9_]', '', col.lower().replace(' ', '_'))
        new_cols.append(new_col)
    df_to_clean.columns = new_cols
    return df_to_clean

# Apply cleaning only if df is not empty (i.e., data was successfully loaded)
if not df.empty:
    df = clean_column_names(df)
    print("Cleaned Column Names:")
    print(df.columns)
else:
    print("Could not clean column names because 'df' is empty or could not be loaded.")

Cleaned Column Names:
Index(['gender', 'raceethnicity', 'parental_level_of_education', 'lunch',
       'test_preparation_course', 'math_score', 'reading_score',
       'writing_score'],
      dtype='object')


#### Handling Missing Values

First, let's check for any missing values in the dataset.

In [7]:
# Check for missing values
print("Missing values per column:")
print(df.isnull().sum())

if df.isnull().sum().sum() == 0:
    print("\nNo missing values found in the dataset. Imputation is not required.")
else:
    print("\nMissing values found. Proceeding with imputation strategies.")

Missing values per column:
gender                         0
raceethnicity                  0
parental_level_of_education    0
lunch                          0
test_preparation_course        0
math_score                     0
reading_score                  0
writing_score                  0
dtype: int64

No missing values found in the dataset. Imputation is not required.


#### Outlier Detection and Neutralization

Next, we will identify and neutralize outliers in the numerical columns using the Z-score method. Outliers will be capped to a certain threshold to minimize their impact on the analysis without removing data points.

In [8]:
from scipy.stats import zscore
import numpy as np

# Define numerical columns for outlier detection
numerical_cols = ['math_score', 'reading_score', 'writing_score']

# Z-score threshold for outlier detection
z_threshold = 3

print("Identifying and neutralizing outliers using Z-scores (threshold = 3):")

for col in numerical_cols:
    # Calculate Z-scores for the column
    df[f'{col}_zscore'] = np.abs(zscore(df[col]))

    # Identify outliers
    outliers = df[df[f'{col}_zscore'] > z_threshold]

    if not outliers.empty:
        print(f"\nOutliers detected in '{col}':")
        print(outliers[[col, f'{col}_zscore']].head())

        # Neutralize outliers by capping (Winsorization)
        # Calculate upper and lower bounds based on non-outlier data
        upper_bound = df[df[f'{col}_zscore'] <= z_threshold][col].max()
        lower_bound = df[df[f'{col}_zscore'] <= z_threshold][col].min()

        df[col] = np.where(df[f'{col}_zscore'] > z_threshold,
                           np.sign(df[col] - df[col].mean()) * upper_bound + df[col].mean() if df[col].mean() < upper_bound else upper_bound,
                           df[col])
        # A simpler capping might be to replace with the max/min of the non-outlier data
        # For positive outliers, replace with max non-outlier value
        df[col] = np.where(df[col] > upper_bound, upper_bound, df[col])
        # For negative outliers, replace with min non-outlier value (if any below current min)
        df[col] = np.where(df[col] < lower_bound, lower_bound, df[col])

        print(f"Outliers in '{col}' have been capped. New descriptive statistics for '{col}':")
        print(df[col].describe())
    else:
        print(f"\nNo significant outliers detected in '{col}' with a Z-score threshold of {z_threshold}.")

    # Drop the temporary zscore column
    df = df.drop(columns=[f'{col}_zscore'])

print("\nOutlier neutralization complete for numerical columns.")

Identifying and neutralizing outliers using Z-scores (threshold = 3):

Outliers detected in 'math_score':
     math_score  math_score_zscore
17           18           3.173040
59            0           4.360728
787          19           3.107058
980           8           3.832867
Outliers in 'math_score' have been capped. New descriptive statistics for 'math_score':
count    1000.00000
mean       66.13200
std        15.01386
min        22.00000
25%        57.00000
50%        66.00000
75%        77.00000
max       100.00000
Name: math_score, dtype: float64

Outliers detected in 'reading_score':
     reading_score  reading_score_zscore
59              17              3.574960
327             23              3.163801
596             24              3.095274
980             24              3.095274
Outliers in 'reading_score' have been capped. New descriptive statistics for 'reading_score':
count    1000.00000
mean       69.18500
std        14.54938
min        26.00000
25%        59.00000


#### Feature Engineering

Now, let's engineer at least 3 new predictive features from the existing data columns to provide more insights and potentially improve model performance.

In [9]:
# 1. Average Score: Mean of math, reading, and writing scores
df['average_score'] = (df['math_score'] + df['reading_score'] + df['writing_score']) / 3

# 2. Total Score: Sum of math, reading, and writing scores
df['total_score'] = df['math_score'] + df['reading_score'] + df['writing_score']

# 3. Has Test Preparation Course: Convert 'test_preparation_course' into a binary numerical feature
df['has_test_prep'] = df['test_preparation_course'].apply(lambda x: 1 if x == 'completed' else 0)

# 4. Performance Category: Categorize students based on their average score
def categorize_performance(score):
    if score >= 80:
        return 'high'
    elif score >= 60:
        return 'medium'
    else:
        return 'low'
df['performance_category'] = df['average_score'].apply(categorize_performance)

# 5. Parental Education Numeric: Map parental level of education to numerical values
education_mapping = {
    "some high school": 0,
    "high school": 1,
    "some college": 2,
    "associate's degree": 3,
    "bachelor's degree": 4,
    "master's degree": 5
}
df['parental_education_numeric'] = df['parental_level_of_education'].map(education_mapping)

print("New features engineered. Displaying first 5 rows with new features:")
print(df.head())

print("\nInformation about the DataFrame with new features:")
df.info()

New features engineered. Displaying first 5 rows with new features:
   gender raceethnicity parental_level_of_education         lunch  \
0  female       group B           bachelor's degree      standard   
1  female       group C                some college      standard   
2  female       group B             master's degree      standard   
3    male       group A          associate's degree  free/reduced   
4    male       group C                some college      standard   

  test_preparation_course  math_score  reading_score  writing_score  \
0                    none        72.0           72.0           74.0   
1               completed        69.0           90.0           88.0   
2                    none        90.0           95.0           93.0   
3                    none        47.0           57.0           44.0   
4                    none        76.0           78.0           75.0   

   average_score  total_score  has_test_prep performance_category  \
0      72.666667     

### Dataset Size and Features Analysis

**Dataset Size:**
- The dataset contains **1000 rows** (entries).
- It has **8 columns** (features).

**Features (Columns) and their Datatypes:**

**Numerical Features:**
- `math score` (int64)
- `reading score` (int64)
- `writing score` (int64)

**Categorical Features:**
- `gender` (object)
- `race/ethnicity` (object)
- `parental level of education` (object)
- `lunch` (object)
- `test preparation course` (object)

All columns have 1000 non-null values, indicating a complete dataset without missing entries.

### What the Data Represents

This dataset appears to contain **Student Performance** information, likely used to analyze factors influencing student scores in various subjects. Each row represents a single student, and the columns provide different attributes about that student and their academic performance.

Here's a breakdown of what each column likely represents:

*   **`gender`**: The gender of the student (e.g., female, male).
*   **`race/ethnicity`**: The student's racial or ethnic group, possibly categorized into different groups (e.g., group A, group B, group C, group D, group E).
*   **`parental level of education`**: The highest level of education attained by one of the student's parents (e.g., bachelor's degree, some college, high school).
*   **`lunch`**: Indicates the type of lunch the student receives, which could be an indicator of socioeconomic status (e.g., standard, free/reduced).
*   **`test preparation course`**: Whether the student completed a test preparation course or not (e.g., none, completed).
*   **`math score`**: The student's score in a math test, ranging from 0 to 100.
*   **`reading score`**: The student's score in a reading test, ranging from 0 to 100.
*   **`writing score`**: The student's score in a writing test, ranging from 0 to 100.

In essence, this dataset allows for exploration into how demographic factors, parental education, and test preparation impact student performance across different subjects.